# QSAR Deep Learning — Transformer Comparison

ChemBERTa-77M-MLM vs ChemBERTa-77M-MTR vs ChemBERTa-druglike vs top 3 classical models.

## 0. Setup

In [ ]:
import pandas as pd, numpy as np, torch, math, os, pickle, warnings
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from rdkit import Chem
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, roc_auc_score, precision_score,
                             recall_score, f1_score)
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          get_cosine_schedule_with_warmup)
from tqdm.notebook import trange
warnings.filterwarnings('ignore')
if torch.cuda.is_available(): device = torch.device('cuda')
elif torch.backends.mps.is_available(): device = torch.device('mps')
else: device = torch.device('cpu')
print(f'Device: {device}')

## 1. Load + binarize

In [ ]:
df = pd.read_csv('data/curated_data_rdkit_Imane.csv')
df.rename(columns={'logC50':'log_LC50','SMILES_curated':'SMILES'}, inplace=True)
meta = df['CAS'].astype(str).str.contains('Meta', na=False)
df = df[~meta].reset_index(drop=True)
df['toxic'] = (df['log_LC50'] > 1.0).astype(int)
pos = df['toxic'].sum(); neg = len(df) - pos
print(f'Molecules: {len(df)} | Toxic: {pos} ({pos/len(df)*100:.1f}%) | Non-toxic: {neg} ({neg/len(df)*100:.1f}%)')
y_all = torch.tensor(df['toxic'].values, dtype=torch.float32)
smiles_all = df['SMILES'].tolist()
ix = np.arange(len(df))
train_idx, test_idx = train_test_split(ix, test_size=0.2, random_state=42, stratify=df['toxic'])
y_train, y_test = y_all[train_idx], y_all[test_idx]
smi_train = [smiles_all[i] for i in train_idx]
smi_test  = [smiles_all[i] for i in test_idx]
print(f'Train: {len(smi_train)}  Test: {len(smi_test)}')

## 2. Dataset + DataLoader

In [ ]:
class SMILESDataset(Dataset):
    def __init__(self, sm, y, tok, ml=256):
        self.enc = tok(sm, padding='max_length', truncation=True, max_length=ml, return_tensors='pt')
        self.y = y
    def __len__(self): return len(self.y)
    def __getitem__(self, i): return {k: v[i] for k,v in self.enc.items()}, self.y[i]
def collate_smiles(b):
    return {k: torch.stack([b[i][0][k] for i in range(len(b))]) for k in b[0][0]}, torch.stack([b[i][1] for i in range(len(b))])
BS = 16  # small batch for MPS memory

## 3. Training + evaluation functions

In [ ]:
def train_transformer(model, tok, smi_tr, y_tr, smi_te, y_te,
                         epochs=30, lr=5e-5, patience=10, name='Model'):
    tr_ds = SMILESDataset(smi_tr, y_tr, tok)
    te_ds = SMILESDataset(smi_te, y_te, tok)
    tr_loader = DataLoader(tr_ds, BS, shuffle=True, collate_fn=collate_smiles)
    te_loader = DataLoader(te_ds, BS, shuffle=False, collate_fn=collate_smiles)
    crit = nn.BCEWithLogitsLoss()
    # Phase 1: head only
    for p in model.roberta.parameters(): p.requires_grad = False
    opt = torch.optim.AdamW([p for n,p in model.named_parameters() if 'classifier' in n], lr=1e-3, weight_decay=1e-5)
    print(f'  Phase 1 (head only)...')
    for ep in range(5):
        model.train(); tl = 0; n = 0
        for b in tr_loader:
            x = {k: v.to(device) for k,v in b[0].items()}; y = b[1].to(device).unsqueeze(-1)
            opt.zero_grad(); l = crit(model(**x).logits, y); l.backward(); opt.step()
            tl += l.item() * len(y); n += len(y)
        model.eval(); vl = 0; nv = 0
        with torch.no_grad():
            for b in te_loader:
                x = {k: v.to(device) for k,v in b[0].items()}; y = b[1].to(device).unsqueeze(-1)
                vl += crit(model(**x).logits, y).item() * len(y); nv += len(y)
        print(f'    {ep+1}: train={tl/n:.4f} val={vl/nv:.4f}')
    # Phase 2: full fine-tune
    for p in model.roberta.parameters(): p.requires_grad = True
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-5)
    ts = epochs * len(tr_loader); wu = int(0.1 * ts)
    sch = get_cosine_schedule_with_warmup(opt, wu, ts)
    best_loss = float('inf'); best_state = None; pc = 0; gs = 0
    print(f'  Phase 2 (full fine-tune)...')
    pbar = trange(epochs, desc=name, leave=True)
    for ep in pbar:
        model.train(); tl = 0; n = 0
        for b in tr_loader:
            x = {k: v.to(device) for k,v in b[0].items()}; y = b[1].to(device).unsqueeze(-1)
            opt.zero_grad(); l = crit(model(**x).logits, y)
            l.backward(); torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            opt.step(); sch.step(); gs += 1
            tl += l.item() * len(y); n += len(y)
        model.eval(); vl = 0; nv = 0
        with torch.no_grad():
            for b in te_loader:
                x = {k: v.to(device) for k,v in b[0].items()}; y = b[1].to(device).unsqueeze(-1)
                vl += crit(model(**x).logits, y).item() * len(y); nv += len(y)
        pbar.set_postfix({'train':f'{tl/n:.4f}','val':f'{vl/nv:.4f}'})
        if vl < best_loss: best_loss=vl; best_state={k:v.clone() for k,v in model.state_dict().items()}; pc=0
        else: pc+=1
        if pc>=patience: pbar.close(); print(f'    Early stop @ {ep+1}'); break
    if best_state: model.load_state_dict(best_state)
    # Eval
    preds, targets = [], []
    model.eval()
    with torch.no_grad():
        for b in te_loader:
            x = {k: v.to(device) for k,v in b[0].items()}; y = b[1]
            preds.extend(torch.sigmoid(model(**x).logits).squeeze(-1).cpu().numpy()); targets.extend(y.cpu().numpy())
    preds = np.array(preds); targets = np.array(targets); binary = (preds > 0.5).astype(int)
    return {'accuracy': accuracy_score(targets, binary), 'auc_roc': roc_auc_score(targets, preds),
            'sensitivity': recall_score(targets, binary, zero_division=0),
            'specificity': recall_score(1-targets, 1-binary, zero_division=0),
            'precision': precision_score(targets, binary, zero_division=0),
            'f1': f1_score(targets, binary, zero_division=0)}

## 4. ChemBERTa-77M-MLM

In [ ]:
print('=== ChemBERTa-77M-MLM ===')
tok_mlm = AutoTokenizer.from_pretrained('DeepChem/ChemBERTa-77M-MLM')
model_mlm = AutoModelForSequenceClassification.from_pretrained(
    'DeepChem/ChemBERTa-77M-MLM', num_labels=1, problem_type='regression', ignore_mismatched_sizes=True)
for p in model_mlm.roberta.requires_grad_(False).parameters(): pass  # ensure all frozen
class SimpleHead(nn.Module):
    def __init__(self, hd): super().__init__(); self.net = nn.Sequential(nn.Dropout(0.3), nn.Linear(hd,1))
    def forward(self, x): return self.net(x[:, 0, :])
model_mlm.classifier = SimpleHead(model_mlm.config.hidden_size)
model_mlm = model_mlm.to(device)
res_mlm = train_transformer(model_mlm, tok_mlm, smi_train, y_train, smi_test, y_test, name='ChemBERTa-MLM')
print('MLM:', {k: f'{v:.4f}' for k,v in res_mlm.items()})

## 5. ChemBERTa-77M-MTR

In [ ]:
print('\n=== ChemBERTa-77M-MTR ===')
tok_mtr = AutoTokenizer.from_pretrained('DeepChem/ChemBERTa-77M-MTR')
model_mtr = AutoModelForSequenceClassification.from_pretrained(
    'DeepChem/ChemBERTa-77M-MTR', num_labels=1, problem_type='regression', ignore_mismatched_sizes=True)
for p in model_mtr.roberta.parameters(): p.requires_grad = False
model_mtr.classifier = SimpleHead(model_mtr.config.hidden_size)
model_mtr = model_mtr.to(device)
res_mtr = train_transformer(model_mtr, tok_mtr, smi_train, y_train, smi_test, y_test, name='ChemBERTa-MTR')
print('MTR:', {k: f'{v:.4f}' for k,v in res_mtr.items()})

## 6. ChemBERTa-druglike

In [ ]:
print('\n=== ChemBERTa-druglike ===')
tok_dl = AutoTokenizer.from_pretrained('Derify/ChemBERTa-druglike')
model_dl = AutoModelForSequenceClassification.from_pretrained(
    'Derify/ChemBERTa-druglike', num_labels=1, problem_type='regression', ignore_mismatched_sizes=True)
for p in model_dl.roberta.parameters(): p.requires_grad = False
model_dl.classifier = SimpleHead(model_dl.config.hidden_size)
model_dl = model_dl.to(device)
res_dl = train_transformer(model_dl, tok_dl, smi_train, y_train, smi_test, y_test, name='ChemBERTa-DL')
print('Druglike:', {k: f'{v:.4f}' for k,v in res_dl.items()})

## 7. Load classical benchmark

In [ ]:
ck = pickle.load(open('models/classical_benchmark.pkl','rb'))
classical_res = {r['model']: r for r in ck['results']}
top3_names = ck['top3']
print(f'Top 3 classical: {top3_names}')
for n in top3_names:
    r = classical_res[n]
    print(f'  {n:25s} AUC={r["auc_roc"]:.4f}  Acc={r["accuracy"]:.4f}  F1={r["f1"]:.4f}')

## 8. Comparison: Transformers vs Top-3 Classical

In [ ]:
trans_res = {'ChemBERTa-MLM': res_mlm, 'ChemBERTa-MTR': res_mtr, 'ChemBERTa-DL': res_dl}
combined = {}
for n in top3_names: combined[f'Classical: {n}'] = classical_res[n]
for n, r in trans_res.items(): combined[f'{n}'] = r
dfc = pd.DataFrame(combined).T
metrics = ['auc_roc','accuracy','f1','sensitivity','specificity','precision']
print(dfc[metrics].to_string())
print(f'\nBest AUC-ROC: {dfc["auc_roc"].idxmax()} ({dfc["auc_roc"].max():.4f})')

## 9. Plot

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
x = np.arange(len(dfc)); colors = ['#4C72B0','#DD8452','#55A868','#C44E52','#937860','#8C8C8C']
for i, (ax, m, t) in enumerate(zip(axes, ['auc_roc','accuracy','f1'],
    ['AUC-ROC (higher is better)','Accuracy (higher is better)','F1 Score (higher is better)'])):
    vals = dfc[m]; order = np.argsort(vals)[::-1]
    bars = ax.bar(x[order], vals.iloc[order], 0.6, color=[colors[j%6] for j in order], edgecolor='black', lw=0.5)
    ax.set_xticks(x[order]); ax.set_xticklabels(dfc.index[order], rotation=25, ha='right', fontsize=8)
    ax.set_title(t, fontsize=11, fontweight='bold'); ax.set_ylim(0, 1.05)
    for bar, val in zip(bars, vals.iloc[order]):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.008, f'{val:.3f}', ha='center', va='bottom', fontsize=7)
plt.tight_layout()
plt.savefig('figures/transformer_vs_classical.png', dpi=150, bbox_inches='tight')
plt.show()

## 10. Save

In [ ]:
os.makedirs('models', exist_ok=True)
torch.save(model_mlm.state_dict(), 'models/ChemBERTa_MLM_binary.pth')
torch.save(model_mtr.state_dict(), 'models/ChemBERTa_MTR_binary.pth')
torch.save(model_dl.state_dict(), 'models/ChemBERTa_DL_binary.pth')
pickle.dump({'MLM':res_mlm,'MTR':res_mtr,'Druglike':res_dl,'classical':classical_res,'top3':top3_names},
            open('models/transformer_results.pkl','wb'))
print('Saved.')